# AI Quality Inspection for Manufactured Parts: run everything on Colab

This notebook downloads the real dataset, generates synthetic training images from the **CAD model**, trains **TensorFlow/Keras** and **PyTorch** models, exports **TFLite**, and builds the comparison report.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Total time is about 1 hour 15 minutes.

| Step | Time on a T4 GPU |
|---|---|
| 1–3 Setup + data | ~3 min |
| 4 CAD synthetic data | ~10 min |
| 5 Train 3 experiments × 2 frameworks | ~60 min |
| 6–8 TFLite, report, measurement check | ~5 min |

## 1. Get the project
Upload `ai-quality-inspection.zip` (Files panel on the left → upload), **or** clone it from your GitHub.

In [ ]:
import os
if os.path.exists('/content/ai-quality-inspection.zip'):
    !unzip -q -o /content/ai-quality-inspection.zip -d /content
else:
    !git clone https://github.com/YOUR-USERNAME/ai-quality-inspection.git /content/ai-quality-inspection
%cd /content/ai-quality-inspection

## 2. Install the extra libraries
TensorFlow and PyTorch are already on Colab. CadQuery is the CAD kernel.

In [ ]:
!pip install -q cadquery
!pip install -q -U kaggle          # new version understands the new Kaggle API tokens
import tensorflow as tf, torch
print('TensorFlow', tf.__version__, '| PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 3. Download the real dataset from Kaggle
Run the cell. A box appears under it:
- Type **1** and press Enter if you have the **new API token** (text starting with `KGAT_`). Then paste it into the next box and press Enter. It stays hidden while you paste.
- Type **2** and press Enter if you downloaded **kaggle.json** (legacy key). Then click **Choose Files** and select it.


In [ ]:
import os, getpass
from google.colab import files
kdir = os.path.expanduser('~/.kaggle'); os.makedirs(kdir, exist_ok=True)
if not (os.path.exists(f'{kdir}/kaggle.json') or os.path.exists(f'{kdir}/access_token')):
    choice = input('Type 1 = paste new API token, 2 = upload kaggle.json, then press Enter: ').strip()
    if choice == '1':
        token = getpass.getpass('Paste your Kaggle API token and press Enter: ').strip()
        with open(f'{kdir}/access_token', 'w') as f:
            f.write(token)
        os.chmod(f'{kdir}/access_token', 0o600)
        os.environ['KAGGLE_API_TOKEN'] = token
    else:
        up = files.upload()                       # choose kaggle.json
        name = next(iter(up))                     # works even if the file was renamed, e.g. "kaggle (1).json"
        with open(f'{kdir}/kaggle.json', 'wb') as f:
            f.write(up[name])
        os.chmod(f'{kdir}/kaggle.json', 0o600)
    print('Kaggle credentials saved.')
!python scripts/download_data.py


## 4. Generate synthetic training images from the CAD model
300 good + 300 defective parts × 4 photos each = 2,400 images, plus sample STEP files you can open in any CAD tool.

In [ ]:
!python scripts/generate_cad_data.py --ok-parts 300 --def-parts 300 --views 4

In [ ]:
import matplotlib.pyplot as plt, glob, cv2
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, cls in enumerate(['ok_front', 'def_front']):
    for ax, p in zip(axes[row], sorted(glob.glob(f'data/synthetic/{cls}/*.jpeg'))[::40][:6]):
        ax.imshow(cv2.imread(p, 0), cmap='gray'); ax.set_title(os.path.basename(p)[:22], fontsize=8); ax.axis('off')
plt.suptitle('CAD-rendered training images (top: good, bottom: defective)'); plt.show()

## 5. Train: 3 training sets × 2 frameworks
All six models are tested on the same **real** test photos.
- `real`: real photos only (baseline)
- `cad_only`: CAD images only (zero real photos)
- `cad_plus_10`: CAD images + 10% of the real photos

Short on time? Run only the baseline: add `--experiments real`.

In [ ]:
!python scripts/run_experiments.py

## 6. Export the Keras model to TensorFlow Lite (edge devices)

In [ ]:
!python scripts/export_tflite.py --experiment real

## 7. Comparison report

In [ ]:
!python scripts/compare.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/comparison.md').read()))
display(Image('results/comparison.png'))

In [ ]:
for fw in ['keras', 'pytorch']:
    print(fw); display(Image(f'results/real/{fw}/history.png')); display(Image(f'results/real/{fw}/gradcam_examples.png'))

## 8. CAD dimension check accuracy (bore vs drawing tolerance)

In [ ]:
!python scripts/measure_check.py --parts 50
display(Image('results/measurement_examples.png'))

## 9. Download your trained models and results
Unzip into the project folder on your PC to run the API and the Streamlit app locally.

In [ ]:
!zip -q -r trained_models_and_results.zip models results
files.download('trained_models_and_results.zip')

## Your numbers for the resume and interview
Open `results/comparison.md`. Note the **defect recall** and **accuracy** for each row, the **TFLite size reduction**, and the **bore measurement error**, then fill them into the interview manual.